In [1]:
import pandas as pd
import xlrd

In [2]:
# ─────────────────────────────────────────────
# 1. LOAD TABLES FROM EXCEL
# ─────────────────────────────────────────────
# Each sheet in the Excel file corresponds to one table.
# Adjust sheet names to match your actual file.
 
credit_applications = pd.read_excel(r'E:\Claude_files\HSBC_Conversion\data\credit_applications.xls')   # columns: relationship_id, credit_proposal_serial_no, app_status, app_proposal_type, app_approval_dt
basic_details = pd.read_excel(r'E:\Claude_files\HSBC_Conversion\data\basic_details.xls')   # columns: relationship_id, customer_id, credit_proposal_serial_no, final_crr_basd, final_pd_basd, internal_score_card_basd
multi_ratings  = pd.read_excel(r'E:\Claude_files\HSBC_Conversion\data\multi_ratings.xls')    # columns: relationship_id, customer_id, credit_proposal_serial_no, final_crr_mrdp, final_pd_mrdp, internal_score_card_mrdp
downstream  = pd.read_excel(r'E:\Claude_files\HSBC_Conversion\data\downstream.xls')    # columns: relationship_id, customer_id, credit_proposal_serial_no, local_customer_id

In [3]:
# ─────────────────────────────────────────────
# 2. CREAP FILTERS
# ─────────────────────────────────────────────
 
# Step 1: Keep only Approved proposals
credit_applications = credit_applications[credit_applications["app_status"] == "A"].copy()
 
# Step 2: Parse approval date and sort to get latest proposal per relationship
credit_applications["app_approval_dt"] = pd.to_datetime(
    credit_applications["app_approval_dt"], dayfirst=True, errors="coerce"
)
credit_applications = credit_applications.sort_values("app_approval_dt", ascending=False)
 
# Step 3: Keep only the latest proposal per relationship_id
credit_applications = credit_applications.drop_duplicates(subset=["relationship_id"], keep="first").copy()
 
# Step 4: Drop records where app_proposal_type is CN
credit_applications = credit_applications[credit_applications["app_proposal_type"] != "CN"].copy()
 
print(f"CREAP rows after filters : {len(credit_applications)}")

CREAP rows after filters : 6


In [4]:
credit_applications

,relationship_id,credit_proposal_serial_no,app_status,app_proposal_type,app_approval_dt
6,1000305,260215,A,NL,2026-02-15
2,1000101,260101,A,RN,2026-01-01
18,20205688,250630,A,RN,2025-06-30
8,FR100201,241205,A,RN,2024-12-05
15,20100374,230901,A,NL,2023-09-01
3,1000202,230415,A,NL,2023-04-15


In [5]:
# ─────────────────────────────────────────────
# 3. JOINS
# ─────────────────────────────────────────────
 
# credit_applications ← inner join → basic_details  (on relationship + serial; customer_id comes in from basic_details)
df = credit_applications.merge(
    basic_details,
    on=["relationship_id", "credit_proposal_serial_no"],
    how="inner",
    suffixes=("", "_basic_details_dup")
)
 
dup_cols = [c for c in df.columns if c.endswith("_basic_details_dup")]
df.drop(columns=dup_cols, inplace=True)
 
print(f"After credit_applications inner join basic_details : {len(df)} rows")
 
 
# credit_applications+basic_details ← left join → multi_ratings  (on relationship + customer + serial)
df = df.merge(
    multi_ratings,
    on=["relationship_id", "customer_id", "credit_proposal_serial_no"],
    how="left",
    suffixes=("", "_multi_ratings_dup")
)
 
dup_cols = [c for c in df.columns if c.endswith("_multi_ratings_dup")]
df.drop(columns=dup_cols, inplace=True)
 
print(f"After left join multi_ratings: {len(df)} rows")
 
 
# credit_applications+basic_details+multi_ratings ← left join → downstream  (on relationship + customer only)
df = df.merge(
    downstream,
    on=["relationship_id", "customer_id"],
    how="left",
    suffixes=("", "_downstream_dup")
)
 
dup_cols = [c for c in df.columns if c.endswith("_downstream_dup")]
df.drop(columns=dup_cols, inplace=True)
 
print(f"After left join downstream: {len(df)} rows")
df

After credit_applications inner join basic_details : 6 rows
After left join multi_ratings: 6 rows
After left join downstream: 6 rows


,relationship_id,credit_proposal_serial_no,app_status,app_proposal_type,app_approval_dt,customer_id,final_crr_basic_details,final_pd_basic_details,internal_score_card_basic_details,final_crr_multi_ratings,final_pd_multi_ratings,internal_score_card_multi_ratings,local_customer_id
0,1000305,260215,A,NL,2026-02-15,1000305,5.1,0.0450,GMIS-1031,5.1,0.0450,GMIS-1031,NaN
1,1000101,260101,A,RN,2026-01-01,1000101,4.5,0.0250,No-GMIS,NaN,NaN,NaN,LCL-IN-4401
2,20205688,250630,A,RN,2025-06-30,20205688,3.2,0.0014,EXT-DE02,3.2,0.0014,EXT-DE02,LCL-DE-3357
3,FR100201,241205,A,RN,2024-12-05,FR100201,3.0,0.0015,EXT-FR01,NaN,NaN,NaN,LCL-FR-7721
4,20100374,230901,A,NL,2023-09-01,20100374,3.4,0.0030,EXT-DE01,3.4,0.0030,EXT-DE01,LCL-DE-2201
5,1000202,230415,A,NL,2023-04-15,1000202,2.8,0.0005,GMIS-1022,2.8,0.0005,GMIS-1022,LCL-IN-5530


In [6]:
# ─────────────────────────────────────────────
# 4. COALESCE: multi_ratings → fallback to basic_details
# ─────────────────────────────────────────────
# If multi_ratings value is missing, use basic_details value instead.
 
df["final_crr"] = df["final_crr_multi_ratings"].combine_first(df["final_crr_basic_details"])
df["final_pd"]  = df["final_pd_multi_ratings"].combine_first(df["final_pd_basic_details"])
df["internal_score_card"] = (
    df["internal_score_card_multi_ratings"].combine_first(df["internal_score_card_basic_details"])
)
df

,relationship_id,credit_proposal_serial_no,app_status,app_proposal_type,app_approval_dt,customer_id,final_crr_basic_details,final_pd_basic_details,internal_score_card_basic_details,final_crr_multi_ratings,final_pd_multi_ratings,internal_score_card_multi_ratings,local_customer_id,final_crr,final_pd,internal_score_card
0,1000305,260215,A,NL,2026-02-15,1000305,5.1,0.0450,GMIS-1031,5.1,0.0450,GMIS-1031,NaN,5.1,0.0450,GMIS-1031
1,1000101,260101,A,RN,2026-01-01,1000101,4.5,0.0250,No-GMIS,NaN,NaN,NaN,LCL-IN-4401,4.5,0.0250,No-GMIS
2,20205688,250630,A,RN,2025-06-30,20205688,3.2,0.0014,EXT-DE02,3.2,0.0014,EXT-DE02,LCL-DE-3357,3.2,0.0014,EXT-DE02
3,FR100201,241205,A,RN,2024-12-05,FR100201,3.0,0.0015,EXT-FR01,NaN,NaN,NaN,LCL-FR-7721,3.0,0.0015,EXT-FR01
4,20100374,230901,A,NL,2023-09-01,20100374,3.4,0.0030,EXT-DE01,3.4,0.0030,EXT-DE01,LCL-DE-2201,3.4,0.0030,EXT-DE01
5,1000202,230415,A,NL,2023-04-15,1000202,2.8,0.0005,GMIS-1022,2.8,0.0005,GMIS-1022,LCL-IN-5530,2.8,0.0005,GMIS-1022


In [7]:
 
# ─────────────────────────────────────────────
# 5. CLEAN UP COLUMN ORDER
# ─────────────────────────────────────────────
 
OUTPUT_COLS = [
    # identifiers
    "relationship_id",
    "customer_id",
    "credit_proposal_serial_no",
    # proposal info
    "app_status",
    "app_proposal_type",
    "app_approval_dt",
    # final resolved risk fields
    "final_crr",
    "final_pd",
    "internal_score_card",
    # WDRP
    "local_customer_id",
]

# Only keep columns that actually exist (safe guard)
output_cols_present = [c for c in OUTPUT_COLS if c in df.columns]
df_out = df[output_cols_present].reset_index(drop=True)
df_out

,relationship_id,customer_id,credit_proposal_serial_no,app_status,app_proposal_type,app_approval_dt,final_crr,final_pd,internal_score_card,local_customer_id
0,1000305,1000305,260215,A,NL,2026-02-15,5.1,0.0450,GMIS-1031,NaN
1,1000101,1000101,260101,A,RN,2026-01-01,4.5,0.0250,No-GMIS,LCL-IN-4401
2,20205688,20205688,250630,A,RN,2025-06-30,3.2,0.0014,EXT-DE02,LCL-DE-3357
3,FR100201,FR100201,241205,A,RN,2024-12-05,3.0,0.0015,EXT-FR01,LCL-FR-7721
4,20100374,20100374,230901,A,NL,2023-09-01,3.4,0.0030,EXT-DE01,LCL-DE-2201
5,1000202,1000202,230415,A,NL,2023-04-15,2.8,0.0005,GMIS-1022,LCL-IN-5530
